<a href="https://colab.research.google.com/github/sananousheenedu/AwaamiAgent/blob/main/RAG_TESTING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pypdf requests pandas numpy faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.8 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/sananousheenedu/AwaamiAgent.git

Cloning into 'AwaamiAgent'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 153 (delta 28), reused 1 (delta 1), pack-reused 98 (from 1)
Receiving objects: 100% (153/153), 62.96 KiB | 4.20 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [3]:
%cd AwaamiAgent

/content/AwaamiAgent


In [4]:
!find . -maxdepth 3 -type f | sort

./app.py
./build_knowledge_base.py
./data/documents/.gitkeep
./data/download_sources.py
./data/sources.csv
./.git/config
./.git/description
./.git/HEAD
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/sendemail-validate.sample
./.git/hooks/update.sample
./.git/index
./.git/info/exclude
./.git/logs/HEAD
./.git/packed-refs
./rag/chunker.py
./rag/document_loader.py
./rag/embeddings.py
./rag/__init__.py
./rag/knowledge_base.py
./rag/rag_pipeline.py
./rag/retrieval.py
./rag/source_mapper.py
./rag/vector_store.py
./requirements.txt
./tesseract-ocr


In [5]:
!python data/download_sources.py

Downloading: NEPRA Consumer Service Manual Revised 2025
Saved: /content/AwaamiAgent/data/documents/ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
Downloading: FESCO Customer Services
Failed: FESCO Customer Services
Error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Downloading: NEPRA Complaint Management System
Failed: NEPRA Complaint Management System
Error: 404 Client Error: Not Found for url: https://nepra.org.pk/CMS/
Downloading: Complaint Handling Rules 2015
Failed: Complaint Handling Rules 2015
Error: 404 Client Error: Not Found for url: https://nepra.org.pk/Legislation/1-Reg/1.1%20Complaint%20Handling%20Rules%202015.pdf
Downloading: Supply of Electric Power Regulations 2015
Failed: Supply of Electric Power Regulations 2015
Error: 404 Client Error: Not Found for url: https://nepra.org.pk/Legislation/2-Reg/2.1%20Supply%20of%20Electric%20Power%20Regulations%202015.pdf
Downloading: Prosumer Regulations 2026
Saved: /content/AwaamiAgent/data/

In [6]:
import os

files = os.listdir("data/documents")

pdf_files = [
    f for f in files
    if f.lower().endswith(".pdf")
]

print("PDF files downloaded:", len(pdf_files))

for f in pdf_files:
    print(f)

PDF files downloaded: 3
ELEC011_SRO 1330_I__2026.pdf
ELEC008_Prosumer Regulations 2026.pdf
ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf


In [7]:
import os

pdf_files = [
    f for f in os.listdir("data/documents")
    if f.lower().endswith(".pdf")
]

print(len(pdf_files))

3


In [8]:
import os

for f in os.listdir("data/documents"):
    if f.lower().endswith(".pdf"):
        print(f)

ELEC011_SRO 1330_I__2026.pdf
ELEC008_Prosumer Regulations 2026.pdf
ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf


In [9]:
import csv

with open("data/sources.csv", "r", encoding="utf-8") as f:
    sources = list(csv.DictReader(f))

downloaded = {
    f.split("_")[0]
    for f in __import__("os").listdir("data/documents")
    if f.lower().endswith(".pdf")
}

print("Missing sources:")
for source in sources:
    if source["source_id"] not in downloaded:
        print(source["source_id"], "-", source["source_name"])

Missing sources:
ELEC002 - FESCO Customer Services
ELEC003 - NEPRA Complaint Management System
ELEC006 - Complaint Handling Rules 2015
ELEC007 - Supply of Electric Power Regulations 2015
ELEC009 - Performance Standards Distribution Regulations 2026
ELEC010 - SRO 547(I)/2026
ELEC012 - SRO 693(I)/2026


In [10]:
!python data/download_sources.py

Downloading: NEPRA Consumer Service Manual Revised 2025
Saved: /content/AwaamiAgent/data/documents/ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
Downloading: FESCO Customer Services
Failed: FESCO Customer Services
Error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Downloading: NEPRA Complaint Management System
Failed: NEPRA Complaint Management System
Error: 404 Client Error: Not Found for url: https://nepra.org.pk/CMS/
Downloading: Complaint Handling Rules 2015
Failed: Complaint Handling Rules 2015
Error: 404 Client Error: Not Found for url: https://nepra.org.pk/Legislation/1-Reg/1.1%20Complaint%20Handling%20Rules%202015.pdf
Downloading: Supply of Electric Power Regulations 2015
Failed: Supply of Electric Power Regulations 2015
Error: 404 Client Error: Not Found for url: https://nepra.org.pk/Legislation/2-Reg/2.1%20Supply%20of%20Electric%20Power%20Regulations%202015.pdf
Downloading: Prosumer Regulations 2026
Saved: /content/AwaamiAgent/data/

In [11]:
!python -c "import requests; u='https://nepra.org.pk/Legislation/2-Rules/2.11%20NEPRA%20Complaint%20Handling%20and%20Dispute%20Resolution%20%28Procedure%29%20Rules%2C%202015/NEPRA%27s%20Compalint%20Handling%20and%20Dispute%20Resolution%20%28Procedure%29%20Rules%202105.pdf'; r=requests.get(u,headers={'User-Agent':'Mozilla/5.0'},timeout=60); print(r.status_code, r.headers.get('Content-Type'), len(r.content), r.content[:10])"

200 application/pdf 493701 b'%PDF-1.4\r\n'


In [12]:
import requests

u = "https://nepra.org.pk/Legislation/2-Rules/2.2%20Supply%20of%20Electric%20Power%20Regulations%2C%202015/NEPRA%27s%20Supply%20of%20Electric%20Power%20Regulations%202015.pdf"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.content[:10])

404
text/html; charset=iso-8859-1
9
b'Not Found'


In [13]:
import requests

u = "https://www.nepra.org.pk/Legislation/3-Reg/3.14%20NEPRA%20%28Supply%20of%20Electric%20Power%29%20Regulations%2C%202015/SRO%201134%28I%292015%20Nepra%20Supply%20of%20Electric%20Power%20Regulations%202015.pdf"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.content[:10])

200
application/pdf
143849
b'%PDF-1.4\r\n'


In [14]:
import requests

u = "https://www.nepra.org.pk/Legislation/3-Reg/3.36%20Performance%20Standards%20Distribution%20Regulations%202026.pdf"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.content[:10])

404
text/html; charset=iso-8859-1
9
b'Not Found'


In [15]:
import requests

url = "https://nepra.org.pk/Legal.php"

r = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print("Page downloaded:", len(r.text))

200
Page downloaded: 128522


In [17]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://nepra.org.pk/Legal.php"

r = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

soup = BeautifulSoup(r.text, "html.parser")

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = urljoin(url, link["href"])

    if "Performance Standards" in text or "1430" in text:
        print("TEXT:", text)
        print("URL:", href)
        print()

In [18]:
import re

matches = re.findall(
    r'href=["\']([^"\']*1430[^"\']*)["\']',
    r.text,
    re.IGNORECASE
)

print("Matches found:", len(matches))

for m in matches:
    print(m)

Matches found: 1
Legislation/3-Reg/3.39%20NEPRA%20Performance%20Standards%20(Distribution)%20Regulations,%202026/SRO%201430(I)-2026%20Dated%2025-08-2026.pdf


In [19]:
import requests

u = "https://nepra.org.pk/Legislation/3-Reg/3.39%20NEPRA%20Performance%20Standards%20(Distribution)%20Regulations,%202026/SRO%201430(I)-2026%20Dated%2025-08-2026.pdf"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.content[:10])

200
application/pdf
2758044
b'%PDF-1.4\n%'


In [20]:
matches = re.findall(
    r'href=["\']([^"\']*547[^"\']*)["\']',
    r.text,
    re.IGNORECASE
)

print("Matches found:", len(matches))

for m in matches:
    print(m)

Matches found: 0


In [21]:
import re

# Find the section around SRO 547 in the downloaded NEPRA page
match = re.search(
    r'.{0,1000}547\(I\)/2026.{0,2000}',
    r.text,
    re.IGNORECASE | re.DOTALL
)

if match:
    print(match.group(0))
else:
    print("SRO 547 text not found in this page")

SRO 547 text not found in this page


In [22]:
import requests

u = "https://nepra.org.pk/Legislation/3-Reg/3.35%20NEPRA%20%20Prosumer%20Regulations%202026/S.R.O%20547%281%29%202026%20notification%20amendment%20in%20NEPRA%20%28Prosumer%29%20Regulations%2002-04-2026.PDF"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.content[:10])

200
application/pdf
188905
b'%PDF-1.4\n%'


In [23]:
matches = re.findall(
    r'href=["\']([^"\']*693[^"\']*)["\']',
    r.text,
    re.IGNORECASE
)

print("Matches found:", len(matches))

for m in matches:
    print(m)

Matches found: 0


In [24]:
import re

# Search for the section containing "Technical Standards for Grid Connectivity"
matches = re.findall(
    r'href=["\']([^"\']*)["\'][^>]*>[^<]*Technical Standards for Grid Connectivity',
    r.text,
    re.IGNORECASE
)

print("Matches found:", len(matches))

for m in matches:
    print(m)

Matches found: 0


In [25]:
import requests

u = "https://nepra.org.pk/Legislation/3-Reg/3.38National%20Electric%20Power%20Regulatory%20Authority%20%28Technical%20Standards%20for%20Grid%20Connectivity%29%20Regulations%202026/SRO%20693%28I%29-2026%20Dated%2027-04-2026%20Grid%20Conncetivity%20Regulations.pdf"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.content[:10])

200
application/pdf
127782
b'%PDF-1.4\n%'


In [26]:
import requests

u = "https://fesco.com.pk/cServices"

# Configure a session with modern browser headers to avoid being blocked
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
})

try:
    r = session.get(u, timeout=60)
    print("Status Code:", r.status_code)
    print("Content-Type:", r.headers.get("Content-Type"))
    print("Length:", len(r.content))
    print(r.text[:200])
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
    print("Note: The server might be blocking connections from Colab's IP address range or requiring Javascript evaluation.")

ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [27]:
import requests

u = "https://fesco.com.pk/cServices"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.text[:200])

ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [28]:
import requests

u = "https://nepra.org.pk/CAD-Database/CMS-CAD/home.php"

r = requests.get(
    u,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(len(r.content))
print(r.text[:200])

200
text/html; charset=UTF-8
20089
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "https://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="https://www.w3.org/1999/xhtml">
<head>
    <meta charset="utf-8"


In [29]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.50 KiB | 1.50 MiB/s, done.
From https://github.com/sananousheenedu/AwaamiAgent
   e296594..09561b4  main       -> origin/main
Updating e296594..09561b4
Fast-forward
 data/sources.csv | 20 ++++++++++----------
 1 file changed, 10 insertions(+), 10 deletions(-)


In [30]:
!python data/download_sources.py

Downloading: NEPRA Consumer Service Manual Revised 2025
Saved: /content/AwaamiAgent/data/documents/ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
Downloading: FESCO Customer Services
Failed: FESCO Customer Services
Error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Downloading: NEPRA Complaint Management System
Skipped: NEPRA Complaint Management System - not a PDF
Downloading: Complaint Handling Rules 2015
Saved: /content/AwaamiAgent/data/documents/ELEC006_Complaint Handling Rules 2015.pdf
Downloading: Supply of Electric Power Regulations 2015
Saved: /content/AwaamiAgent/data/documents/ELEC007_Supply of Electric Power Regulations 2015.pdf
Downloading: Prosumer Regulations 2026
Saved: /content/AwaamiAgent/data/documents/ELEC008_Prosumer Regulations 2026.pdf
Downloading: Performance Standards Distribution Regulations 2026
Failed: Performance Standards Distribution Regulations 2026
Error: 404 Client Error: Not Found for url: https://nepra.org.pk

In [31]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1021 bytes | 1021.00 KiB/s, done.
From https://github.com/sananousheenedu/AwaamiAgent
   09561b4..316c776  main       -> origin/main
Updating 09561b4..316c776
Fast-forward
 data/sources.csv | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [32]:
!python data/download_sources.py

Downloading: NEPRA Consumer Service Manual Revised 2025
Saved: /content/AwaamiAgent/data/documents/ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
Downloading: FESCO Customer Services
Failed: FESCO Customer Services
Error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Downloading: NEPRA Complaint Management System
Skipped: NEPRA Complaint Management System - not a PDF
Downloading: Complaint Handling Rules 2015
Saved: /content/AwaamiAgent/data/documents/ELEC006_Complaint Handling Rules 2015.pdf
Downloading: Supply of Electric Power Regulations 2015
Saved: /content/AwaamiAgent/data/documents/ELEC007_Supply of Electric Power Regulations 2015.pdf
Downloading: Prosumer Regulations 2026
Saved: /content/AwaamiAgent/data/documents/ELEC008_Prosumer Regulations 2026.pdf
Downloading: Performance Standards Distribution Regulations 2026
Saved: /content/AwaamiAgent/data/documents/ELEC009_Performance Standards Distribution Regulations 2026.pdf
Downloading: SRO 

In [33]:
import os

pdf_files = [
    f for f in os.listdir("data/documents")
    if f.lower().endswith(".pdf")
]

print("PDF files downloaded:", len(pdf_files))
for f in sorted(pdf_files):
    print("✅", f)

PDF files downloaded: 8
✅ ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
✅ ELEC006_Complaint Handling Rules 2015.pdf
✅ ELEC007_Supply of Electric Power Regulations 2015.pdf
✅ ELEC008_Prosumer Regulations 2026.pdf
✅ ELEC009_Performance Standards Distribution Regulations 2026.pdf
✅ ELEC010_SRO 547_I__2026.pdf
✅ ELEC011_SRO 1330_I__2026.pdf
✅ ELEC012_SRO 693_I__2026.pdf


In [34]:
!python build_knowledge_base.py

Traceback (most recent call last):
  File "/content/AwaamiAgent/build_knowledge_base.py", line 6, in <module>
    from rag.chunker import prepare_chunks
ImportError: cannot import name 'prepare_chunks' from 'rag.chunker' (/content/AwaamiAgent/rag/chunker.py)


In [35]:
!grep -n "^def " rag/chunker.py

23:def load_source_metadata() -> Dict[str, Dict]:
65:def extract_pdf_text(pdf_path: str) -> List[Dict]:
94:def load_documents() -> List[Dict]:


In [36]:
!cat rag/chunker.py

import csv
import os
from typing import List, Dict

from pypdf import PdfReader


BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

SOURCES_FILE = os.path.join(
    BASE_DIR,
    "data",
    "sources.csv"
)

DOCUMENTS_DIR = os.path.join(
    BASE_DIR,
    "data",
    "documents"
)


def load_source_metadata() -> Dict[str, Dict]:
    """
    Load source information from data/sources.csv.

    Returns:
        Dictionary where the key is source_id.
    """

    sources = {}

    if not os.path.exists(SOURCES_FILE):
        raise FileNotFoundError(
            f"Sources file not found: {SOURCES_FILE}"
        )

    with open(
        SOURCES_FILE,
        "r",
        encoding="utf-8-sig"
    ) as file:

        reader = csv.DictReader(file)

        for row in reader:
            source_id = row.get("source_id", "").strip()

            if not source_id:
                continue

            sources[source_id] = {
                "source_id": source_id,
           

In [37]:
!cat build_knowledge_base.py

import os
import sys
import subprocess

from rag.document_loader import load_documents
from rag.chunker import prepare_chunks
from rag.vector_store import build_and_save_vector_store


BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DOWNLOAD_SCRIPT = os.path.join(BASE_DIR, "data", "download_sources.py")
DOCUMENTS_DIR = os.path.join(BASE_DIR, "data", "documents")


def main():
    print("=" * 60)
    print("AwaamiAgent Knowledge Base Builder")
    print("=" * 60)

    # Step 1: Download official source documents
    print("\n[1/4] Downloading official civic documents...")

    if not os.path.exists(DOWNLOAD_SCRIPT):
        raise FileNotFoundError(
            f"Download script not found: {DOWNLOAD_SCRIPT}"
        )

    subprocess.run(
        [sys.executable, DOWNLOAD_SCRIPT],
        check=True
    )

    # Step 2: Load PDF documents
    print("\n[2/4] Loading PDF documents...")

    documents = load_documents()

    if not documents:
        raise RuntimeError(
            "N

In [38]:
!head -40 rag/document_loader.py

import csv
import os
from typing import List, Dict

from pypdf import PdfReader


BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

SOURCES_FILE = os.path.join(
    BASE_DIR,
    "data",
    "sources.csv"
)

DOCUMENTS_DIR = os.path.join(
    BASE_DIR,
    "data",
    "documents"
)


def load_source_metadata() -> Dict[str, Dict]:
    """
    Load source information from data/sources.csv.

    Returns:
        Dictionary where the key is source_id.
    """

    sources = {}

    if not os.path.exists(SOURCES_FILE):
        raise FileNotFoundError(
            f"Sources file not found: {SOURCES_FILE}"
        )

    with open(
        SOURCES_FILE,
        "r",


In [39]:
%%writefile rag/chunker.py

import re
from typing import List, Dict


DEFAULT_CHUNK_SIZE = 1200
DEFAULT_CHUNK_OVERLAP = 200


def normalize_text(text: str) -> str:
    """
    Normalize extracted PDF text.
    """
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def chunk_text(
    text: str,
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    chunk_overlap: int = DEFAULT_CHUNK_OVERLAP
) -> List[str]:
    """
    Split text into overlapping character-based chunks.
    """

    text = normalize_text(text)

    if not text:
        return []

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller than chunk_size"
        )

    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:

        end = min(
            start + chunk_size,
            text_length
        )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = end - chunk_overlap

    return chunks


def prepare_chunks(
    documents: List[Dict],
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    chunk_overlap: int = DEFAULT_CHUNK_OVERLAP
) -> List[Dict]:
    """
    Convert loaded document pages into searchable text chunks.

    Each chunk preserves the original document metadata.
    """

    chunks = []

    for document in documents:

        text = document.get("text", "")
        metadata = document.get("metadata", {}).copy()

        text_chunks = chunk_text(
            text,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_number, chunk in enumerate(
            text_chunks,
            start=1
        ):

            chunk_metadata = metadata.copy()

            chunk_metadata["chunk"] = chunk_number

            chunks.append({
                "text": chunk,
                "metadata": chunk_metadata
            })

    return chunks


if __name__ == "__main__":

    print("Chunker module loaded successfully.")

Overwriting rag/chunker.py


In [40]:
from rag.chunker import prepare_chunks
print("✅ prepare_chunks imported successfully")

✅ prepare_chunks imported successfully


In [41]:
!python build_knowledge_base.py

AwaamiAgent Knowledge Base Builder

[1/4] Downloading official civic documents...
Downloading: NEPRA Consumer Service Manual Revised 2025
Saved: /content/AwaamiAgent/data/documents/ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
Downloading: FESCO Customer Services
Failed: FESCO Customer Services
Error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Downloading: NEPRA Complaint Management System
Skipped: NEPRA Complaint Management System - not a PDF
Downloading: Complaint Handling Rules 2015
Saved: /content/AwaamiAgent/data/documents/ELEC006_Complaint Handling Rules 2015.pdf
Downloading: Supply of Electric Power Regulations 2015
Saved: /content/AwaamiAgent/data/documents/ELEC007_Supply of Electric Power Regulations 2015.pdf
Downloading: Prosumer Regulations 2026
Saved: /content/AwaamiAgent/data/documents/ELEC008_Prosumer Regulations 2026.pdf
Downloading: Performance Standards Distribution Regulations 2026
Saved: /content/AwaamiAgent/data/documents

In [42]:
!grep -n -A50 "def build_and_save_vector_store" rag/vector_store.py

213:def build_and_save_vector_store(
214-    chunks: List[Dict],
215-    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
216-) -> CivicVectorStore:
217-    """
218-    Build and save a civic FAISS vector store.
219-    """
220-
221-    store = CivicVectorStore(
222-        model_name=model_name
223-    )
224-
225-    store.build(
226-        chunks
227-    )
228-
229-    store.save()
230-
231-    return store
232-
233-
234-def load_vector_store(
235-    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
236-) -> CivicVectorStore:
237-    """
238-    Load an existing civic vector store.
239-    """
240-
241-    store = CivicVectorStore(
242-        model_name=model_name
243-    )
244-
245-    store.load()
246-
247-    return store
248-
249-
250-if __name__ == "__main__":
251-
252-    print(
253-        "Civic vector store module loaded successfully."
254-    )


In [43]:
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.04 KiB | 535.00 KiB/s, done.
From https://github.com/sananousheenedu/AwaamiAgent
   316c776..285a43c  main       -> origin/main
Updating 316c776..285a43c
Fast-forward
 build_knowledge_base.py | 18 ++++++++++++------
 1 file changed, 12 insertions(+), 6 deletions(-)


In [44]:
!python build_knowledge_base.py

AwaamiAgent Knowledge Base Builder

[1/4] Downloading official civic documents...
Downloading: NEPRA Consumer Service Manual Revised 2025
Saved: /content/AwaamiAgent/data/documents/ELEC001_NEPRA Consumer Service Manual Revised 2025.pdf
Downloading: FESCO Customer Services
Failed: FESCO Customer Services
Error: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Downloading: NEPRA Complaint Management System
Skipped: NEPRA Complaint Management System - not a PDF
Downloading: Complaint Handling Rules 2015
Saved: /content/AwaamiAgent/data/documents/ELEC006_Complaint Handling Rules 2015.pdf
Downloading: Supply of Electric Power Regulations 2015
Saved: /content/AwaamiAgent/data/documents/ELEC007_Supply of Electric Power Regulations 2015.pdf
Downloading: Prosumer Regulations 2026
Saved: /content/AwaamiAgent/data/documents/ELEC008_Prosumer Regulations 2026.pdf
Downloading: Performance Standards Distribution Regulations 2026
Saved: /content/AwaamiAgent/data/documents

In [45]:
import os

print("Vector store files:")

for f in os.listdir("data/vector_store"):
    path = os.path.join("data/vector_store", f)
    print(f"✅ {f} — {os.path.getsize(path):,} bytes")

Vector store files:
✅ civic.index — 671,277 bytes
✅ metadata.json — 700,220 bytes


In [46]:
from rag.rag_pipeline import query_civic_rag

result = query_civic_rag(
    "What should I do if I have a complaint about my electricity connection or billing?",
    top_k=5
)

print("QUESTION:")
print(result["question"])

print("\n" + "=" * 60)
print("RETRIEVED SOURCES")
print("=" * 60)

for source in result["sources"]:
    print(source)

print("\n" + "=" * 60)
print("GROUNDED CONTEXT")
print("=" * 60)

print(result["context"][:6000])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUESTION:
What should I do if I have a complaint about my electricity connection or billing?

RETRIEVED SOURCES
{'source_id': 'ELEC001', 'category': 'electricity', 'source_name': 'NEPRA Consumer Service Manual Revised 2025', 'source_url': 'https://nepra.org.pk/Legislation/7-Manuals/2025/CONSUMER%20SERVICE%20MANUAL%20%28CSM%29%20REVISED%202025.pdf', 'authority': 'NEPRA', 'description': 'Official consumer service rules and procedures for electricity consumers', 'last_verified': '2026-10-04'}
{'source_id': 'ELEC009', 'category': 'electricity', 'source_name': 'Performance Standards Distribution Regulations 2026', 'source_url': 'https://nepra.org.pk/Legislation/3-Reg/3.39%20NEPRA%20Performance%20Standards%20(Distribution)%20Regulations,%202026/SRO%201430(I)-2026%20Dated%2025-08-2026.pdf', 'authority': 'NEPRA', 'description': 'Official performance standards applicable to electricity distribution companies', 'last_verified': '2026-10-04'}

GROUNDED CONTEXT
[EVIDENCE 1]

Source ID: ELEC001
Sou

In [48]:
from rag.rag_pipeline import query_civic_rag

result = query_civic_rag(
    "My electricity bill has an arithmetic error. Where should I complain and how quickly should it be corrected?",
    top_k=5
)

print("QUESTION:")
print(result["question"])

print("\n" + "=" * 60)
print("SOURCES")
print("=" * 60)

for source in result["sources"]:
    print(
        source["source_id"],
        "|",
        source["source_name"]
    )

print("\n" + "=" * 60)
print("EVIDENCE")
print("=" * 60)

print(result["context"][:5000])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUESTION:
My electricity bill has an arithmetic error. Where should I complain and how quickly should it be corrected?

SOURCES
ELEC001 | NEPRA Consumer Service Manual Revised 2025

EVIDENCE
[EVIDENCE 1]

Source ID: ELEC001
Source Name: NEPRA Consumer Service Manual Revised 2025
Authority: NEPRA
Category: electricity
Page: 66
Chunk: 1
Similarity Score: 0.5599
Last Verified: 2026-10-04
Official URL: https://nepra.org.pk/Legislation/7-Manuals/2025/CONSUMER%20SERVICE%20MANUAL%20%28CSM%29%20REVISED%202025.pdf

Evidence Text:
*
*
(h) The 1st Bill for a new connection shall be issued within two months of the 
date of connection. Such complaints need to be redressed within fifteen days. 
If delivered later, it should have appropriate slab corresponding to the number 
of months for which the accumulated reading bill is delivered.
(i) Inclusion of paid amounts in next month's bills. Revenue Officers (ROs) as 
well as Bank Managers, Incharge Post Office where the last bills were paid 
are compet

In [49]:
import os

print("Vector store files:")
for file in os.listdir("data/vector_store"):
    path = os.path.join("data/vector_store", file)
    print(file, "→", os.path.getsize(path), "bytes")

Vector store files:
civic.index → 671277 bytes
metadata.json → 700220 bytes


In [50]:
from google.colab import files

files.download("data/vector_store/civic.index")
files.download("data/vector_store/metadata.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 732.69 KiB | 5.86 MiB/s, done.
From https://github.com/sananousheenedu/AwaamiAgent
   285a43c..23b6306  main       -> origin/main
Updating 285a43c..23b6306
error: The following untracked working tree files would be overwritten by merge:
	data/vector_store/civic.index
	data/vector_store/metadata.json
Please move or remove them before you merge.
Aborting


In [52]:
!rm -f data/vector_store/civic.index data/vector_store/metadata.json
!git pull

Updating 285a43c..23b6306
Fast-forward
 data/vector_store/.gitkeep      |    1 +
 data/vector_store/civic.index   |  Bin 0 -> 671277 bytes
 data/vector_store/metadata.json | 6557 +++++++++++++++++++++++++++++++++++++++
 3 files changed, 6558 insertions(+)
 create mode 100644 data/vector_store/.gitkeep
 create mode 100644 data/vector_store/civic.index
 create mode 100644 data/vector_store/metadata.json


In [53]:
from rag.rag_pipeline import query_civic_rag

result = query_civic_rag(
    "What should I do if my electricity bill contains an arithmetic error?",
    top_k=3
)

print("QUESTION:")
print(result["question"])

print("\n" + "=" * 60)
print("SOURCES")
print("=" * 60)

for source in result["sources"]:
    print(
        source["source_id"],
        "|",
        source["source_name"],
        "| Page:",
        source.get("page"),
        "| Score:",
        source.get("similarity_score")
    )

print("\n" + "=" * 60)
print("GROUNDED CONTEXT")
print("=" * 60)

print(result["context"][:3500])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUESTION:
What should I do if my electricity bill contains an arithmetic error?

SOURCES
ELEC001 | NEPRA Consumer Service Manual Revised 2025 | Page: None | Score: None

GROUNDED CONTEXT
[EVIDENCE 1]

Source ID: ELEC001
Source Name: NEPRA Consumer Service Manual Revised 2025
Authority: NEPRA
Category: electricity
Page: 66
Chunk: 1
Similarity Score: 0.5036
Last Verified: 2026-10-04
Official URL: https://nepra.org.pk/Legislation/7-Manuals/2025/CONSUMER%20SERVICE%20MANUAL%20%28CSM%29%20REVISED%202025.pdf

Evidence Text:
*
*
(h) The 1st Bill for a new connection shall be issued within two months of the 
date of connection. Such complaints need to be redressed within fifteen days. 
If delivered later, it should have appropriate slab corresponding to the number 
of months for which the accumulated reading bill is delivered.
(i) Inclusion of paid amounts in next month's bills. Revenue Officers (ROs) as 
well as Bank Managers, Incharge Post Office where the last bills were paid 
are competent 